In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/ferdousjannatul/
      /kaggle/input/datasets/ferdousjannatul/resultsrfdetr/
        data.yaml
        rtdetr-l.pt
        yolo26n.pt
        yolov8m.pt
        /kaggle/input/datasets/ferdousjannatul/resultsrfdetr/dataset/
        /kaggle/input/datasets/ferdousjannatul/resultsrfdetr/runs/
        /kaggle/input/datasets/ferdousjannatul/resultsrfdetr/saved_models/
          RT-DETR-L_best.pt
          YOLOv8m_best.pt
          nb2_results.csv
      /kaggle/input/datasets/ferdousjannatul/resultsyolo11/
        data.yaml
        yolo11m.pt
        yolo11n.pt
        yolo11s.pt
        yolo26n.pt
        /kaggle/input/datasets/ferdousjannatul/resultsyolo11/dataset/
        /kaggle/input/datasets/ferdousjannatul/resultsyolo11/plots/
          eda_01_class_distribution.png
          eda_02_size_geometry.png
          eda_03_density.png
          eda_04_spatial_heatmap.png
          eda_05_sample_grid.png
          eda_06_cooc

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/ferdousjannatul/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
import infer, dataset as ds, train
utils.require_gpu()
assert utils.install_detection_stack()
LEGACY = {"YOLO11n": "YOLO11n_best.pt", "YOLO11s": "YOLO11s_best.pt", "YOLO11m": "YOLO11m_best.pt",
          "YOLOv8m": "YOLOv8m_best.pt", "RT-DETR-L": "RT-DETR-L_best.pt"}
info = ds.materialize("orig")
for model, fn in LEGACY.items():
    found = utils.find_files(fn)
    rid = f"orig__{model}__slegacy"
    if not found:
        print("not attached:", fn); continue
    if train.find_existing(rid)[0]:
        print("already done:", rid); continue
    infer.evaluate_run(rid, model, found[0], info, meta=dict(note="weights of the rejected manuscript", path=found[0]))

GPU: {'device': 'cuda:0', 'gpu': 'Tesla T4', 'n_gpu': 2, 'torch': '2.11.0+cu128'}
materialized 'orig' (0291ae6cd605) -> /kaggle/working/data_orig  images {'train': 1305, 'val': 54, 'test': 55}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1882.1±454.7 MB/s, size: 87.1 KB)
val: Scanning /kaggle/working/data_orig/test/labels... 55 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 55/55 880.7it/s 0.1s
val: New cache created: /kaggle/working/data_orig/test/labels.cache
                 Class     Images  Instances   

In [4]:
# any trained weights whose evaluation is missing (e.g. a session ended during evaluation)
import re
for root, dirs, files in os.walk(cfg.INPUT_ROOT):
    for f in files:
        m = re.match(r"^(orig|grouped)__(.+)__s(\d+)_best\.pt$", f)
        if not m: continue
        rid = f[:-len("_best.pt")]
        if train.find_existing(rid)[0]: continue
        split_name, model = m.group(1), m.group(2)
        print("evaluating missing:", rid)
        infer.evaluate_run(rid, model, os.path.join(root, f), ds.materialize(split_name))

## Cross-dataset test (optional, needs the SkySeaLand runs of 02G and both split Datasets attached)
Models trained on one dataset are tested on the OTHER dataset's grouped test split without fine-tuning.
Run ids `xfer-main2sky__<model>__s42` and `xfer-sky2main__<model>__s42` -> table T6 in notebook 04.

In [5]:
XFER = [("grouped", "sky_grouped", "xfer-main2sky"), ("sky_grouped", "grouped", "xfer-sky2main")]
for src_split, dst_split, tag in XFER:
    try:
        dst = ds.materialize(dst_split)
    except FileNotFoundError as e:
        print("skip", tag, "-", e); continue
    for model in cfg.MODELS:
        w = utils.find_files(f"{src_split}__{model}__s42_best.pt")
        rid = f"{tag}__{model}__s42"
        if not w:
            print("no weights for", src_split, model); continue
        if train.find_existing(rid)[0]:
            print("already done:", rid); continue
        infer.evaluate_run(rid, model, w[0], dst, meta=dict(trained_on=src_split, tested_on=dst_split))

materialized 'sky_grouped' (0b03f1f732a3) -> /kaggle/working/data_sky_grouped  images {'train': 922, 'val': 193, 'test': 192}
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2670.7±555.3 MB/s, size: 268.4 KB)
val: Scanning /kaggle/working/data_sky_grouped/test/labels... 192 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 192/192 1.3Kit/s 0.2s
val: New cache created: /kaggle/working/data_sky_grouped/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 14.1it/s 1.7s
                   all        192       2795      0.712      0.522       0.57      0.255
Speed: 0.4ms preprocess, 4.2ms inference, 0.0ms loss, 1.5ms postprocess per image
[timer] xfer-main2sky__YOLO11n__s42 ultralytics test metrics: 0.0 min (2s)
  xfer-main2sky__YOLO11n__s42  sta

2026-10-20 21:05:50,567 - sahi - WARNING - ignoring invalid prediction with bbox: [6.5074615478515625, 184, 31.887447357177734, 184] (ultralytics.py:278)


  xfer-main2sky__YOLO11n__s42  sahi320   AP50=0.5381 AP=0.2219 AR50_micro=0.542 | conf>=0.25: dets=11417 TP=2039 FP=9378 R=0.730 | 315.0 ms/img
  xfer-main2sky__YOLO11n__s42  sahi160   AP50=0.3942 AP=0.1760 AR50_micro=0.604 | conf>=0.25: dets=18741 TP=1945 FP=16796 R=0.696 | 963.4 ms/img
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2285.0±874.7 MB/s, size: 204.2 KB)
val: Scanning /kaggle/working/data_sky_grouped/test/labels.cache... 192 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 192/192 67.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 12.6it/s 1.9s
                   all        192       2795      0.748      0.619       0.67      0.336
Speed: 0.5ms preprocess, 5.6ms inference, 0.0ms loss, 1.2ms postprocess per image
[timer] xfer-